# M062 Reports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append("../../")

import pyaldata as pyal
import pandas as pd
import numpy as np

from tools.reports.report_initial import run_initial_report

In [ ]:
all_dfs = []

M061: M061_2025_03_04_10_00  M061_2025_03_05_14_00  M061_2025_03_06_14_00

## 21st of March 14:00h

In [ ]:
# Files 
session = 'M061_2025_03_06_14_00'
data_dir = f"/data/bnd-data/raw/M061/{session}"
fname0 = os.path.join(data_dir, f"{session}_pyaldata_0.mat")
fname1 = os.path.join(data_dir, f"{session}_pyaldata_1.mat")
# fname2 = os.path.join(data_dir, f"{session}_pyaldata_2.mat")


# Load files
df0 = pyal.mat2dataframe(fname0, shift_idx_fields=False)
df1 = pyal.mat2dataframe(fname1, shift_idx_fields=False)
# df2 = pyal.mat2dataframe(fname2, shift_idx_fields=False)

all_dfs.append(pd.concat([df0, df1], ignore_index=True))

del df0
del df1
# del df2

In [ ]:
# Files 
session = 'M061_2025_03_05_14_00'
data_dir = f"/data/bnd-data/raw/M061/{session}"
fname0 = os.path.join(data_dir, f"{session}_pyaldata_0.mat")
fname1 = os.path.join(data_dir, f"{session}_pyaldata_1.mat")
# fname2 = os.path.join(data_dir, f"{session}_pyaldata_2.mat")


# Load files
df0 = pyal.mat2dataframe(fname0, shift_idx_fields=False)
df1 = pyal.mat2dataframe(fname1, shift_idx_fields=False)
# df2 = pyal.mat2dataframe(fname2, shift_idx_fields=False)

all_dfs.append(pd.concat([df0, df1], ignore_index=True)) 
del df0
del df1
# del df2

In [ ]:
# Files 
session = 'M061_2025_03_04_10_00'
data_dir = f"/data/bnd-data/raw/M061/{session}"
fname0 = os.path.join(data_dir, f"{session}_pyaldata_0.mat")
fname1 = os.path.join(data_dir, f"{session}_pyaldata_1.mat")
fname2 = os.path.join(data_dir, f"{session}_pyaldata_2.mat")


# Load files
df0 = pyal.mat2dataframe(fname0, shift_idx_fields=False)
df1 = pyal.mat2dataframe(fname1, shift_idx_fields=False)
df2 = pyal.mat2dataframe(fname2, shift_idx_fields=False)

all_dfs.append(pd.concat([df0, df1, df2], ignore_index=True))

del df0
del df1
del df2

In [ ]:
from tools.dsp.preprocessing import preprocess
all_dfs_trials = []
for dataframe in all_dfs:
    all_dfs_trials.append(preprocess(dataframe, only_trials=True, repair_time_varying_fields=['MotSen1_X', 'MotSen1_Y']))


In [ ]:
from tools.dsp.preprocessing import preprocess
all_dfs_trials = []
for dataframe in all_dfs:
    all_dfs_trials.append(preprocess(dataframe, only_trials=True))


In [ ]:
from tools.dsp.preprocessing import preprocess
all_dfs_all = []
for dataframe in all_dfs:
    all_dfs_all.append(preprocess(dataframe,only_trials=False))


In [ ]:
all_dfs_all[1]= all_dfs_all[1].drop(all_dfs_all[1].loc[all_dfs_all[1]['trial_name']=='trial'].tail(1).index).reset_index(drop=True)


In [ ]:
from tools.viz import mean_firing as firing
import matplotlib.pyplot as plt
from tools.params import Params
from tools.decoding import decodeTools as decode

In [ ]:
from tools import params

In [ ]:
df_problem = all_dfs_trials[1]

In [ ]:
for trial in range(len(df_problem)):
    print(df_problem["MOp_rates"][trial].shape)

In [ ]:
all_dfs_trials[0]["MOp_rates"][0].shape

In [ ]:
df_problem["idx_sol_on"]

In [ ]:
all_dfs_trials[0].columns

In [ ]:

all_dfs_trials[1] = all_dfs_trials[1].iloc[:-1]
all_dfs_trials[1] = all_dfs_trials[1].reset_index(drop=True)

In [ ]:
areas=["MOp", "SSp_ul", "CP", "Thal"]
fig, ax = plt.subplots(figsize=(8,5))

decode.plot_decoding_moving_window(
    ax, category="values_Sol_direction", df_list=all_dfs_trials, 
    areas=areas, n_components=30, model="pca",
    idx_event="idx_sol_on", min_time=-1.5, max_time=2,
    window_length=0.1, step=0.03, trial_conditions=[]
)

plt.show()

In [ ]:
from tools.viz import utilityTools as utility
from sklearn.decomposition import PCA

In [ ]:
def plot_pca_projection_spectra(
    sessions,
    areas,
    ref_epoch,
    proj_epochs,
    n_components=10,
    epoch_selectors=None,
):
    """
    Plot PCA variance-explained spectra for multiple areas and projection epochs,
    averaged across sessions, in a 2x2 grid (one subplot per area).
    
    Parameters
    ----------
    sessions : list of pd.DataFrame
        List of session DataFrames (`df_all_trials`).
    areas : list of str
        List of column names in each DataFrame containing rate arrays.
    ref_epoch : str
        Key of the epoch to use for fitting the PCA basis.
    proj_epochs : list of str
        Keys of epochs to project onto the PCA basis.
    n_components : int, optional
        Number of PCA components to compute (default: 10).
    epoch_selectors : dict, optional
        Mapping from epoch key to a function f(df, area) -> 2D array.
        If None, defaults to Free, Intertrial, PostPerturb as before.
    shaded_errorbar_func : function, optional
        Function with signature shaded_errorbar(ax, x, y, ...) plotting mean±error.
        Must be provided.
    """
    # Default epoch selectors
    if epoch_selectors is None:
        epoch_selectors = {
            'Free':      lambda df, area: np.concatenate(pyal.select_trials(df.iloc[:-1], "trial_name=='free'")[area].values, axis=0),
            'Intertrial':lambda df, area: np.concatenate(pyal.select_trials(df, "trial_name=='intertrial'")[area].values, axis=0),
            'PostPerturb':lambda df, area: np.concatenate(
                pyal.restrict_to_interval(
                    pyal.select_trials(df, "trial_name=='trial'"),
                    epoch_fun=Params.perturb_epoch
                )[area].values,
                axis=0
            )
        }

    # Prepare storage
    results = {area: {epoch: [] for epoch in proj_epochs} for area in areas}

    # Loop over sessions and areas
    for df_all in sessions:
        for area in areas:
            # Reference data for PCA
            ref_data = epoch_selectors[ref_epoch](df_all, area)
            pca = PCA(n_components=n_components, svd_solver='full')
            pca.fit(ref_data)
            total_var = np.sum(np.var(ref_data, axis=0))

            # Project and compute variance fractions
            for proj in proj_epochs:
                proj_data = epoch_selectors[proj](df_all, area)
                pc_scores = pca.transform(proj_data)
                var_by_pc = np.var(pc_scores, axis=0)
                frac_var = var_by_pc / total_var
                results[area][proj].append(frac_var)

    # Stack into arrays: shape = (n_components, n_sessions)
    for area in areas:
        for proj in proj_epochs:
            results[area][proj] = np.stack(results[area][proj], axis=1)

    # Plot 2x2 subplots
    fig, axs = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
    x = np.arange(1, n_components + 1)
    linestyles = {proj: style for proj, style in zip(proj_epochs, ['-', '--', ':'])}
    colors = dict(zip(areas, plt.rcParams['axes.prop_cycle'].by_key()['color']))

    for ax, area in zip(axs.flat, areas):
        for proj in proj_epochs:
            y = results[area][proj]  # (PCs x sessions)
            utility.shaded_errorbar(
                ax, x, y,
                lineStat   = np.mean,
                errorStat  = np.std,
                color      = colors[area],
                linestyle  = linestyles.get(proj, '-'),
                label      = f"{area.split('_')[0]}→{proj}"
            )
        ax.set_title(area)
        ax.set_xlabel('PC #')
        ax.set_ylabel('Fraction var explained')
        ax.legend(fontsize='small')

    fig.suptitle(f"PCA ({ref_epoch}-basis) projections across areas")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


In [ ]:
plot_pca_projection_spectra(
    sessions=all_dfs_all,
    areas=["MOp_rates", "SSp_rates", "CP_rates", "VAL_rates"],
    ref_epoch='Free',
    proj_epochs=['Free','Intertrial','PostPerturb'],
    n_components=10,
)

In [ ]:
from scipy.stats import sem

In [ ]:
def plot_pca_projection_spectra(
    sessions,
    areas,
    ref_epoch,
    proj_epochs,
    n_components=10,
    epoch_selectors=None,

):
    """
    Plot PCA variance-explained spectra for multiple areas and projection epochs,
    averaged across sessions, in a 2x2 grid (one subplot per area).

    Parameters
    ----------
    sessions : list of pd.DataFrame
        List of session DataFrames (`df_all_trials`).
    areas : list of str
        List of column names in each DataFrame containing rate arrays.
    ref_epoch : str
        Key of the epoch to use for fitting the PCA basis.
    proj_epochs : list of str
        Keys of epochs to project onto the PCA basis.
    n_components : int, optional
        Number of PCA components to compute (default: 10).
    epoch_selectors : dict, optional
        Mapping from epoch key to a function f(df, area) -> 2D array.
        If None, defaults to Free, Intertrial, PostPerturb as before.
    shaded_errorbar_func : function, optional
        Function with signature shaded_errorbar(ax, x, y, ...) plotting mean±error.
        Must be provided.
    """
    # Default epoch selectors
    if epoch_selectors is None:
        epoch_selectors = {
            'Free':      lambda df, area: np.concatenate(pyal.select_trials(df.iloc[:-1], "trial_name=='free'")[area].values, axis=0),
            'Intertrial':lambda df, area: np.concatenate(pyal.select_trials(df, "trial_name=='intertrial'")[area].values, axis=0),
            'PostPerturb':lambda df, area: np.concatenate(
                pyal.restrict_to_interval(
                    pyal.select_trials(df, "trial_name=='trial'"),
                    epoch_fun=Params.perturb_epoch
                )[area].values,
                axis=0
            )
        }

    # Prepare storage
    results = {area: {epoch: [] for epoch in proj_epochs} for area in areas}

    # Loop over sessions and areas
    for df_all in sessions:
        for area in areas:
            # Reference data for PCA
            ref_data = epoch_selectors[ref_epoch](df_all, area)
            pca = PCA(n_components=n_components, svd_solver='full')
            pca.fit(ref_data)
            total_var = np.sum(np.var(ref_data, axis=0))

            # Project and compute variance fractions
            for proj in proj_epochs:
                proj_data = epoch_selectors[proj](df_all, area)
                pc_scores = pca.transform(proj_data)
                var_by_pc = np.var(pc_scores, axis=0)
                frac_var = var_by_pc / total_var
                results[area][proj].append(frac_var)

    # Stack into arrays: shape = (n_components, n_sessions)
    for area in areas:
        for proj in proj_epochs:
            results[area][proj] = np.stack(results[area][proj], axis=1)

    # Plot 2x2 subplots without sharing Y-axis
    fig, axs = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=False)
    x = np.arange(1, n_components + 1)
    linestyles = {proj: style for proj, style in zip(proj_epochs, ['-', '--', ':'])}
    # colors = dict(zip(areas, plt.rcParams['axes.prop_cycle'].by_key()['color']))

    for ax, area in zip(axs.flat, areas):
        short_name = area.split('_')[0]
        color=getattr(params.colors, short_name, "k")
        for proj in proj_epochs:
            y = results[area][proj]  # (PCs x sessions)
            utility.shaded_errorbar(
                ax, x, y,
                lineStat   = np.mean,
                errorStat  = np.std,
                color      = color,
                linestyle  = linestyles.get(proj, '-'),
                label      = f"{proj}→{ref_epoch}"
            )
        ax.set_title(short_name)
        ax.set_xlabel('PC #')
        ax.set_ylabel('Fraction var explained')
        ax.legend(fontsize='small')

    fig.suptitle(f"PCA ({ref_epoch}-basis) projections across areas")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
plot_pca_projection_spectra(
    sessions=all_dfs_all,
    areas=["MOp_rates", "SSp_ul_rates", "CP_rates", "Thal_rates"],
    ref_epoch='Free',
    proj_epochs=['Free','Intertrial','PostPerturb'],
    n_components=10,
)

In [ ]:
plot_pca_projection_spectra(
    sessions=[all_dfs_all[0]],
    areas=["MOp_rates", "SSp_ul_rates", "CP_rates", "Thal_rates"],
    ref_epoch='Free',
    proj_epochs=['Free','Intertrial','PostPerturb'],
    n_components=10,
)

In [ ]:
plot_pca_projection_spectra(
    sessions=[all_dfs_all[1]],
    areas=["MOp_rates", "SSp_ul_rates", "CP_rates", "Thal_rates"],
    ref_epoch='Free',
    proj_epochs=['Free','Intertrial','PostPerturb'],
    n_components=10,
)

In [ ]:
plot_pca_projection_spectra(
    sessions=[all_dfs_all[2]],
    areas=["MOp_rates", "SSp_ul_rates", "CP_rates", "Thal_rates"],
    ref_epoch='Free',
    proj_epochs=['Free','Intertrial','PostPerturb'],
    n_components=10,
)

In [ ]:
plot_pca_projection_spectra(
    sessions=all_dfs_all,
    areas=["MOp_rates", "SSp_ul_rates", "CP_rates", "Thal_rates"],
    ref_epoch='Intertrial',
    proj_epochs=['Free','Intertrial','PostPerturb'],
    n_components=10,
)

In [ ]:
plot_pca_projection_spectra(
    sessions=all_dfs_all,
    areas=["MOp_rates", "SSp_ul_rates", "CP_rates", "Thal_rates"],
    ref_epoch='PostPerturb',
    proj_epochs=['Free','Intertrial','PostPerturb'],
    n_components=10,
)